In [1]:
import pyspark
from pyspark.sql import SparkSession 

In [ ]:
spark = SparkSession.builder \
    .appName("ContentRecommendationEDA")\
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .getOrCreate()

25/07/21 21:32:11 WARN Utils: Your hostname, vaibhavi-HP-Laptop-15-fd0xxx resolves to a loopback address: 127.0.1.1; using 192.168.0.128 instead (on interface wlo1)
25/07/21 21:32:11 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/21 21:32:12 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
df = spark.read.parquet("/home/vaibhavi/spark-ml-venv/ml_project/book_recommender/data/books_vectorized")
#/home/vaibhavi/spark-ml-venv/ml_project/preprocessing/output


In [4]:
df.printSchema()

root
 |-- Title: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- review_count: long (nullable = true)



In [25]:
from pyspark.sql import functions as F


In [26]:
df_vectorized.printSchema()

root
 |-- Title: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- review_count: long (nullable = true)
 |-- title_tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- title_filtered: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- title_tf: vector (nullable = true)
 |-- title_tfidf: vector (nullable = true)
 |-- author_tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- author_vec: vector (nullable = true)
 |-- category_tokens: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- category_vec: vector (nullable = true)
 |-- review_count_vec: vector (nullable = true)
 |-- review_count_scaled: vector (nullable = true)
 |-- final_features: vector (nullable = true)



In [29]:
from pyspark.sql.functions import col

df_vectorized.filter(col("review_count") > 5000).select("title").show(truncate=False)


+-------------------------------------+
|title                                |
+-------------------------------------+
|Atlas Shrugged                       |
|The Hobbit                           |
|The Great Gatsby                     |
|Brave New World                      |
|Of Mice and Men                      |
|The Giver                            |
|The Picture of Dorian Gray           |
|Persuasion                           |
|Great Expectations                   |
|Pride and Prejudice                  |
|Mere Christianity                    |
|Wuthering Heights                    |
|Harry Potter and The Sorcerer's Stone|
+-------------------------------------+



In [32]:
from pyspark.ml.linalg import Vectors
from pyspark.sql.functions import col, udf
from pyspark.ml.functions import vector_to_array
from pyspark.sql.types import DoubleType
import numpy as np
import pandas as pd

In [37]:
book_vector = df_vectorized.filter(col("Title") == "The Hobbit").select("final_features").first()["final_features"]
broadcast_vec = spark.sparkContext.broadcast(book_vector.toArray())


In [38]:
def cosine_sim(vec):
    if vec is None:
        return 0.0
    vec1 = vec.toArray()
    vec2 = broadcast_vec.value
    dot = float(np.dot(vec1, vec2))
    norm = np.linalg.norm(vec1) * np.linalg.norm(vec2)
    return float(dot / norm) if norm != 0 else 0.0

cosine_udf = udf(cosine_sim, DoubleType())


In [39]:
df_sim = df_vectorized.withColumn("similarity", cosine_udf("final_features"))
df_sim.orderBy(col("similarity").desc()).select("title", "similarity").show(10, truncate=False)


+-----------------------------------------------------+------------------+
|title                                                |similarity        |
+-----------------------------------------------------+------------------+
|The Hobbit                                           |1.0000000000000002|
|Crucible, The                                        |0.9789467997140651|
|Caldecott                                            |0.9783885673884307|
|Q                                                    |0.9783778589111998|
|The Vision                                           |0.9783415580575551|
|Sarkhan                                              |0.9783331754005907|
|Remote                                               |0.9783030761963375|
|Musclebound                                          |0.9783009609842661|
|The Reaches                                          |0.9782770157834391|
|When Marian Sang: The True Recital of Marian Anderson|0.8793333615384179|
+------------------------